In [13]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [14]:
def build_zone_station_map(shapefile_path):
    zones = gpd.read_file(shapefile_path)

    # use correct ID column
    zones = zones.rename(columns={"OBJECTID": "PULocationID"})
    zones["PULocationID"] = zones["PULocationID"].astype(int)

    # project to planar CRS for centroids
    zones = zones.set_geometry("geometry").to_crs(epsg=2263)
    zones["centroid"] = zones.geometry.centroid

    # back to lat/lon
    centroids = gpd.GeoSeries(zones["centroid"], crs="EPSG:2263").to_crs(epsg=4326)
    zones["lon"] = centroids.x
    zones["lat"] = centroids.y

    # station coordinates
    stations = pd.DataFrame({
        "station": ["laguardia", "jfk", "central_park"],
        "lat": [40.7792, 40.6398, 40.7812],
        "lon": [-73.88, -73.7789, -73.9665]
    })

    def nearest_station(lat, lon):
        dists = (
            (stations["lat"] - lat) ** 2 +
            (stations["lon"] - lon) ** 2
        )
        return stations.loc[dists.idxmin(), "station"]

    zones["station"] = zones.apply(
        lambda row: nearest_station(row["lat"], row["lon"]),
        axis=1
    )

    zone_station_map = zones[["PULocationID", "station"]].copy()
    zone_station_map["PULocationID"] = zone_station_map["PULocationID"].astype(int)

    return zone_station_map

In [15]:
zone_station_map = build_zone_station_map("taxi_zones.shp")

print(zone_station_map.shape)
print(zone_station_map["PULocationID"].nunique())
print(zone_station_map["station"].value_counts())

(263, 2)
263
station
central_park    139
laguardia        82
jfk              42
Name: count, dtype: int64


In [16]:
def clean_weather_file(filepath, station_name):
    weather = pd.read_csv(filepath)

    weather["pickup_hour"] = pd.to_datetime(
        weather["DATE"], errors="coerce"
    ).dt.floor("h")

    keep_cols = [
        "pickup_hour",
        "HourlyPrecipitation",
        "HourlyDryBulbTemperature",
        "HourlyWindSpeed",
        "HourlyVisibility"
    ]

    existing_cols = [c for c in keep_cols if c in weather.columns]
    weather = weather[existing_cols].copy()

    weather = weather.rename(columns={
        "HourlyPrecipitation": "precipitation",
        "HourlyDryBulbTemperature": "temperature",
        "HourlyWindSpeed": "wind_speed",
        "HourlyVisibility": "visibility"
    })

    for col in ["precipitation", "temperature", "wind_speed", "visibility"]:
        if col in weather.columns:
            weather[col] = pd.to_numeric(weather[col], errors="coerce")

    weather = (
        weather.groupby("pickup_hour", as_index=False)
        .mean(numeric_only=True)
        .sort_values("pickup_hour")
        .reset_index(drop=True)
    )

    weather["station"] = station_name
    return weather

In [17]:
def build_weather_year(year, laguardia_path, central_park_path, jfk_path):
    lga = clean_weather_file(laguardia_path, "laguardia")
    cp = clean_weather_file(central_park_path, "central_park")
    jfk = clean_weather_file(jfk_path, "jfk")

    weather = pd.concat([lga, cp, jfk], ignore_index=True)

    weather["pickup_hour"] = pd.to_datetime(weather["pickup_hour"])

    full_hours = pd.date_range(
        f"{year}-01-01 00:00:00",
        f"{year}-12-31 23:00:00",
        freq="h"
    )

    stations = sorted(weather["station"].dropna().unique())

    full_index = pd.MultiIndex.from_product(
        [full_hours, stations],
        names=["pickup_hour", "station"]
    )

    weather_full = (
        weather.set_index(["pickup_hour", "station"])
        .reindex(full_index)
        .reset_index()
        .sort_values(["station", "pickup_hour"])
    )

    weather_full["was_missing"] = weather_full["temperature"].isna()

    cols = ["precipitation", "temperature", "wind_speed", "visibility"]

    filled_parts = []
    for station, grp in weather_full.groupby("station", group_keys=False):
        grp = grp.sort_values("pickup_hour").set_index("pickup_hour")

        for col in cols:
            grp[col] = grp[col].interpolate(method="time", limit_direction="both")

        filled_parts.append(grp.reset_index())

    weather_full = pd.concat(filled_parts, ignore_index=True)
    return weather_full

In [18]:
def build_taxi_weather_year(taxi_df, weather_full, zone_station_map):
    taxi = taxi_df.copy()

    taxi["PULocationID"] = taxi["PULocationID"].astype(int)
    taxi["pickup_hour"] = pd.to_datetime(taxi["pickup_hour"]).dt.floor("h")

    zone_station_map = zone_station_map.copy()
    zone_station_map["PULocationID"] = zone_station_map["PULocationID"].astype(int)

    taxi = taxi.drop(columns=["station"], errors="ignore").merge(
        zone_station_map,
        on="PULocationID",
        how="left"
    )

    taxi_weather = taxi.merge(
        weather_full,
        on=["pickup_hour", "station"],
        how="left"
    )

    taxi_weather["rain_category"] = np.select(
        [
            taxi_weather["precipitation"] == 0,
            (taxi_weather["precipitation"] > 0) & (taxi_weather["precipitation"] <= 0.5),
            taxi_weather["precipitation"] > 0.5
        ],
        [0, 1, 2],
        default=0
    )

    taxi_weather = taxi_weather.sort_values(
        ["PULocationID", "pickup_hour"]
    ).reset_index(drop=True)

    taxi_weather["lag_1"] = (
        taxi_weather.groupby("PULocationID")["trip_count"].shift(1)
    )
    taxi_weather["lag_2"] = (
        taxi_weather.groupby("PULocationID")["trip_count"].shift(2)
    )
    taxi_weather["lag_24"] = (
        taxi_weather.groupby("PULocationID")["trip_count"].shift(24)
    )
    taxi_weather["lag_168"] = (
        taxi_weather.groupby("PULocationID")["trip_count"].shift(168)
    )

    return taxi_weather

In [19]:
def load_taxi_year(year):
    filepath = f"yellow_hourly_demand_full_{year}.parquet"
    taxi = pd.read_parquet(filepath)

    taxi["PULocationID"] = taxi["PULocationID"].astype(int)
    taxi["pickup_hour"] = pd.to_datetime(taxi["pickup_hour"]).dt.floor("h")

    return taxi
    
year_configs = {
    2022: {
        "laguardia": "LaGuardia/LCD_USW00014732_2022_HourlyPrecipitation_nonempty.csv",
        "central_park": "Central_park/LCD_USW00094728_2022_HourlyPrecipitation_nonempty.csv",
        "jfk": "JFK_airport/LCD_USW00094789_2022_HourlyPrecipitation_nonempty.csv",
    },
    2023: {
        "laguardia": "LaGuardia/LCD_USW00014732_2023_HourlyPrecipitation_nonempty.csv",
        "central_park": "Central_park/LCD_USW00094728_2023_HourlyPrecipitation_nonempty.csv",
        "jfk": "JFK_airport/LCD_USW00094789_2023_HourlyPrecipitation_nonempty.csv",
    },
    2024: {
        "laguardia": "LaGuardia/LCD_USW00014732_2024_HourlyPrecipitation_nonempty.csv",
        "central_park": "Central_park/LCD_USW00094728_2024_HourlyPrecipitation_nonempty.csv",
        "jfk": "JFK_airport/LCD_USW00094789_2024_HourlyPrecipitation_nonempty.csv",
    },
    2025: {
        "laguardia": "LaGuardia/LCD_USW00014732_2025_HourlyPrecipitation_nonempty.csv",
        "central_park": "Central_park/LCD_USW00094728_2025_HourlyPrecipitation_nonempty.csv",
        "jfk": "JFK_airport/LCD_USW00094789_2025_HourlyPrecipitation_nonempty.csv",
    }
}
results = {}

for year, cfg in year_configs.items():
    print(f"Processing {year}...")

    taxi_df = load_taxi_year(year)

    weather_full = build_weather_year(
        year,
        cfg["laguardia"],
        cfg["central_park"],
        cfg["jfk"]
    )

    taxi_weather = build_taxi_weather_year(
        taxi_df,
        weather_full,
        zone_station_map
    )

    print(f"{year} shape: {taxi_weather.shape}")
    print(f"{year} missing weather: {taxi_weather['temperature'].isna().sum()}")

    taxi_weather.to_parquet(f"taxi_weather_{year}.parquet", index=False)
    results[year] = taxi_weather

Processing 2022...


/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather["pickup_hour"] = pd.to_datetime(
/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:2: DtypeWarning: Columns (0: HourlyVisibility) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.read_csv(filepath)
/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wea

2022 shape: (2303880, 22)
2022 missing weather: 0
Processing 2023...


/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather["pickup_hour"] = pd.to_datetime(
/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather["pickup_hour"] = pd.to_datetime(
/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:2: DtypeWarning: Columns (0: HourlyWindDirection) have mixed types. Specify dtype option on import or set low_memory=

2023 shape: (2303880, 22)
2023 missing weather: 0
Processing 2024...


/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather["pickup_hour"] = pd.to_datetime(
/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather["pickup_hour"] = pd.to_datetime(
/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many tim

2024 shape: (2310192, 22)
2024 missing weather: 0
Processing 2025...


/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather["pickup_hour"] = pd.to_datetime(
/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather["pickup_hour"] = pd.to_datetime(
/var/folders/wp/vmt1ycjj67s7xs_zy5flbqhm0000gn/T/ipykernel_34106/3532009779.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many tim

2025 shape: (2303880, 22)
2025 missing weather: 0


In [22]:
df = pd.read_parquet("taxi_weather_2023.parquet")

print(df.columns.tolist())
display(df.head())
display (df.sample(20))

['PULocationID', 'pickup_hour', 'trip_count', 'date', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'station', 'precipitation', 'temperature', 'wind_speed', 'visibility', 'was_missing', 'rain_category', 'lag_1', 'lag_2', 'lag_24', 'lag_168']


,PULocationID,pickup_hour,trip_count,date,year,month,day,hour,day_of_week,is_weekend,...,precipitation,temperature,wind_speed,visibility,was_missing,rain_category,lag_1,lag_2,lag_24,lag_168
0,1,2023-01-01 00:00:00,0,2023-01-01,2023,1,1,0,6,True,...,0.005,11.7,2.6,9.656,False,1,NaN,NaN,NaN,NaN
1,1,2023-01-01 01:00:00,0,2023-01-01,2023,1,1,1,6,True,...,0.005,11.7,3.6,11.265,False,1,0.0,NaN,NaN,NaN
2,1,2023-01-01 02:00:00,0,2023-01-01,2023,1,1,2,6,True,...,0.000,11.1,2.6,14.484,False,0,0.0,0.0,NaN,NaN
3,1,2023-01-01 03:00:00,0,2023-01-01,2023,1,1,3,6,True,...,0.000,11.7,4.1,16.093,False,0,0.0,0.0,NaN,NaN
4,1,2023-01-01 04:00:00,0,2023-01-01,2023,1,1,4,6,True,...,0.000,11.1,2.1,16.093,False,0,0.0,0.0,NaN,NaN


,PULocationID,pickup_hour,trip_count,date,year,month,day,hour,day_of_week,is_weekend,...,precipitation,temperature,wind_speed,visibility,was_missing,rain_category,lag_1,lag_2,lag_24,lag_168
1656222,190,2023-01-25 06:00:00,0,2023-01-25,2023,1,25,6,2,False,...,0.000,2.2,2.60,16.093,False,0,0.0,0.0,0.0,0.0
1210597,139,2023-03-13 13:00:00,0,2023-03-13,2023,3,13,13,0,False,...,0.800,5.6,5.70,6.437,False,2,0.0,1.0,0.0,0.0
2070965,237,2023-05-31 05:00:00,13,2023-05-31,2023,5,31,5,2,False,...,0.000,11.7,0.00,14.484,False,0,6.0,1.0,10.0,16.0
1196127,137,2023-07-18 15:00:00,77,2023-07-18,2023,7,18,15,1,False,...,0.000,24.4,1.50,11.265,False,0,48.0,54.0,82.0,71.0
209659,24,2023-12-07 19:00:00,11,2023-12-07,2023,12,7,19,3,False,...,0.000,3.3,1.50,14.484,False,0,12.0,18.0,7.0,11.0
534554,62,2023-01-09 02:00:00,0,2023-01-09,2023,1,9,2,0,False,...,0.005,3.3,3.60,12.875,False,1,0.0,0.0,0.0,0.0
570169,66,2023-02-02 01:00:00,0,2023-02-02,2023,2,2,1,3,False,...,0.000,-1.1,2.10,16.093,False,0,0.0,0.0,0.0,0.0
2067743,237,2023-01-16 23:00:00,40,2023-01-16,2023,1,16,23,0,False,...,0.000,3.3,2.10,16.093,False,0,68.0,128.0,53.0,42.0
1550259,177,2023-12-21 03:00:00,1,2023-12-21,2023,12,21,3,3,False,...,0.000,2.8,4.10,16.093,False,0,0.0,0.0,0.0,0.0
940304,108,2023-05-05 08:00:00,1,2023-05-05,2023,5,5,8,4,False,...,0.000,13.9,2.10,16.093,False,0,0.0,0.0,0.0,1.0


In [24]:
import pandas as pd

files = [
    "taxi_weather_2022.parquet",
    "taxi_weather_2023.parquet",
    "taxi_weather_2024.parquet",
    "taxi_weather_2025.parquet"
]

df_all = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print(df_all.shape)
display(df_all.head())
display (df_all.sample(20))



(9221832, 22)


,PULocationID,pickup_hour,trip_count,date,year,month,day,hour,day_of_week,is_weekend,...,precipitation,temperature,wind_speed,visibility,was_missing,rain_category,lag_1,lag_2,lag_24,lag_168
0,1,2022-01-01 00:00:00,0,2022-01-01,2022,1,1,0,5,True,...,0.000,10.6,0.0,14.484,False,0,NaN,NaN,NaN,NaN
1,1,2022-01-01 01:00:00,0,2022-01-01,2022,1,1,1,5,True,...,0.000,10.6,0.0,11.265,False,0,0.0,NaN,NaN,NaN
2,1,2022-01-01 02:00:00,0,2022-01-01,2022,1,1,2,5,True,...,0.000,10.6,2.1,14.484,False,0,0.0,0.0,NaN,NaN
3,1,2022-01-01 03:00:00,0,2022-01-01,2022,1,1,3,5,True,...,0.005,10.0,2.1,11.265,False,1,0.0,0.0,NaN,NaN
4,1,2022-01-01 04:00:00,0,2022-01-01,2022,1,1,4,5,True,...,0.005,10.0,0.0,9.656,False,1,0.0,0.0,NaN,NaN


,PULocationID,pickup_hour,trip_count,date,year,month,day,hour,day_of_week,is_weekend,...,precipitation,temperature,wind_speed,visibility,was_missing,rain_category,lag_1,lag_2,lag_24,lag_168
139593,16,2022-12-08 09:00:00,0,2022-12-08,2022,12,8,9,3,False,...,0.000000,10.000000,4.600000,16.093000,False,0,0.0,0.0,0.0,0.0
6896086,261,2024-07-05 22:00:00,11,2024-07-05,2024,7,5,22,4,False,...,0.000000,24.400000,1.500000,3.219000,False,0,21.0,28.0,21.0,17.0
2972449,77,2023-04-28 01:00:00,0,2023-04-28,2023,4,28,1,4,False,...,0.000000,10.000000,3.600000,16.093000,False,0,0.0,0.0,0.0,0.0
564138,65,2022-05-26 18:00:00,9,2022-05-26,2022,5,26,18,3,False,...,0.000000,18.300000,2.100000,16.093000,False,0,8.0,3.0,4.0,8.0
4088234,204,2023-09-11 02:00:00,0,2023-09-11,2023,9,11,2,0,False,...,0.000000,22.200000,2.100000,4.828000,False,0,0.0,0.0,0.0,0.0
72635,9,2022-04-17 11:00:00,0,2022-04-17,2022,4,17,11,6,True,...,0.000000,8.900000,10.800000,16.093000,False,0,0.0,0.0,0.0,0.0
7942686,117,2025-12-24 06:00:00,0,2025-12-24,2025,12,24,6,2,False,...,0.005000,2.920548,2.702740,16.093000,True,1,0.0,0.0,0.0,0.0
888313,102,2022-05-29 01:00:00,0,2022-05-29,2022,5,29,1,6,True,...,0.000000,17.800000,3.100000,16.093000,False,0,0.0,0.0,0.0,0.0
8633447,196,2025-10-31 23:00:00,0,2025-10-31,2025,10,31,23,4,False,...,0.200000,15.212903,5.269892,15.227946,True,1,1.0,0.0,0.0,0.0
1798906,206,2022-05-10 10:00:00,0,2022-05-10,2022,5,10,10,1,False,...,0.000000,20.600000,4.600000,16.093000,False,0,0.0,0.0,0.0,0.0


In [25]:
df_all.to_parquet("taxi_weather_2022_2025.parquet", index=False)